In [1]:
# Install deps
!pip install pinecone sentence_transformers bson mongoengine


# Env config
%env PINECONE_API_KEY=pcsk_5zmtAT_Hf7BPztEgkqyeNs9Lb9kHoihUxmi3ZKT1CN14gSLwPxryDiPmYGDrT9eoyEkR8z
%env INDEX_NAME=knowledge-base-1
%env OPENROUTER_API_KEY=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
%env OPENROUTER_MODEL=openrouter/aurora-alpha

env: PINECONE_API_KEY=pcsk_5zmtAT_Hf7BPztEgkqyeNs9Lb9kHoihUxmi3ZKT1CN14gSLwPxryDiPmYGDrT9eoyEkR8z
env: INDEX_NAME=knowledge-base-1
env: OPENROUTER_API_KEY=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
env: OPENROUTER_MODEL=openrouter/aurora-alpha


In [2]:
# Intialise a pinecone client (pinecone.py)

import os
from pinecone import Pinecone

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("INDEX_NAME")

pc = Pinecone(api_key=PINECONE_API_KEY)

def get_pinecone_index():
    index_list = pc.list_indexes()
    index_names = [idx["name"] for idx in index_list]
    print("Available Pinecone index names:", index_names)
    if INDEX_NAME not in index_names:
        raise ValueError(f"Index {INDEX_NAME} not found. Available: {index_names}")
    return pc.Index(INDEX_NAME)

print(get_pinecone_index())


Available Pinecone index names: ['knowledge-base-1']


In [3]:
# OpenRouter configuration and run_llm() (llm.py)

import os
import requests

# Openrouter config
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Model config
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL")
OPENROUTER_TEMPERATURE = 0.2
OPENROUTER_MAX_TOKENS = 2000


# Helper to create a tagged prompt for the llm
def _messages_to_tagged_prompt(messages) -> str:
    prompt = ""
    for msg in messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        if role == "system":
            prompt += f"[System]\n{content}\n"
        elif role == "user":
            prompt += f"[User]\n{content}\n"
        elif role == "assistant":
            prompt += f"[Assistant]\n{content}\n"
        else:
            prompt += f"[{role}]\n{content}\n"
    return prompt.strip()


def run_llm(messages):
    if not OPENROUTER_API_KEY:
        raise RuntimeError("OPENROUTER_API_KEY is not set.")

    prompt = _messages_to_tagged_prompt(messages)

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "Polaris",
    }

    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": OPENROUTER_TEMPERATURE,
        "max_tokens": OPENROUTER_MAX_TOKENS
    }

    response = requests.post(
        OPENROUTER_URL,
        headers=headers,
        json=payload,
        timeout=60
    )

    if response.status_code != 200:
        raise RuntimeError(f"{response.status_code} status code from OpenRouter: {response.text}")

    data = response.json()
    return data["choices"][0]["message"]["content"]

In [4]:
# Helper to create a prompt for llm (prompt.py)

def build_prompt(context_chunks, user_query):
    context_str = "\n---\n".join(context_chunks)

    return f"""
--- RELEVANT CV CONTEXT ---
{context_str or "No context found"}
--- END OF CONTEXT ---

User Question: {user_query}
"""


In [5]:
# Initalise embedding model (embeddings.py)

import os
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"

try:
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cpu")
except Exception as e:
    print("Error loading embedding model:", e)
    embedding_model = None


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# Context retreival logic (context_retrieval.py)

def retrieve_context(user_query: str, user_id: int):
    if embedding_model is None:
        raise RuntimeError("Embedding model not initialized.")

    # Generate embedding vector
    vector = embedding_model.encode(user_query).tolist()
    index = get_pinecone_index()

    # Query Pinecone
    result = index.query(
        vector=vector,
        top_k=10,
        include_metadata=True,
        # filter={"user_id": {"$eq": user_id}}
    )

    chunks = []
    sources = []

    for m in result.matches:
        print("Raw metadata:", type(m.metadata), m.metadata)

        md = m.metadata

        # Convert metadata to normal dict safely
        if hasattr(md, "to_dict"):
            md = md.to_dict()
        elif not isinstance(md, dict):
            md = dict(md) if md else {}

        md = md or {}


        chunks.append(md.get("text", ""))
        sources.append({
            "id": m.id,
            "score": m.score,
            # "cv_url": md.get("cv_url", "N/A"),
        })

    print("Chunks:", chunks)
    print("Sources:", sources)

    return chunks, sources


In [7]:
# Function to call prior helpers and trigger rag for model response (rag_service.py)

from bson import ObjectId

def generate_ai_response(chat_id: str, user_query: str):
    chat = []
    user_id = 1

    # Retrieve relevant context
    context_chunks, source_metadata = retrieve_context(user_query, user_id)

    # Fetch last 5 messages in chronological order
    history = []

    messages = [
        {
            "role": "system",
            "content": "You are an expert recruitment assistant. Answer user queries based on context."
        }
    ]

    # Add message history
    for h in history:
        messages.append({
            "role": "user" if h.sender == 0 else "assistant",
            "content": h.content,
        })

    # Add current user query with context
    messages.append({
        "role": "user",
        "content": build_prompt(context_chunks, user_query)
    })

    # Generate AI response
    reply = run_llm(messages)

    return reply, {"retrieved_sources": source_metadata}


In [8]:
# Test RAG pipline behavior

ai_reply_content, source_context = generate_ai_response(
    chat_id="1",
    user_query="What skills do i need to be an LCVP in FIN?"
)

print("ai_reply_content:", ai_reply_content)
print("source_context:", source_context)

Available Pinecone index names: ['knowledge-base-1']
Raw metadata: <class 'dict'> {'chunk_index': 1, 'competency_id': 'fin_soft_management', 'competency_name': 'Management', 'file_name': 'fin_competencies.json', 'function_code': 'FIN', 'function_name': 'Finance', 'skill_category': 'Functional', 'skill_type': 'Soft', 'source_sheet': 'FIN Functional Skills', 'source_workbook': 'Competency Models.xlsx', 'text': ' and internal audits.\n- Expert: Designs finance systems, coaches finance teams, ensures sustainability.\nRole Level Requirements:\n- Level 2 : TL (Team Leader): Intermediate\n- Level 3 : LCVP (Local Committee Vice-President) / EST (Entity Support Team): Advanced\n- Level 4 : MCVP (Member Committee Vice-President) / GST (Global Support Team): Advanced\n- Level 4 : AI (AIESEC International): Expert\nSource: Competency Models.xlsx / FIN (Finance) Functional Skills'}
Raw metadata: <class 'dict'> {'chunk_index': 1, 'competency_id': 'fin_hard_cash_flow_management', 'competency_name': '